In [ ]:
%pylab inline
import pandas as pd
import scanpy as sc

In [ ]:
def postprocess(adata,view=True):
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata)
    adata.raw = adata
    adata = adata[:, adata.var.highly_variable]
    print('after select HVG',adata.shape)
    sc.pp.scale(adata)
    sc.tl.pca(adata)
    sc.pp.neighbors(adata)
    if view:
        sc.tl.umap(adata)
    return adata

In [ ]:
def cluster_k_leiden(embadata,n_cluster):
    max_steps=120
    this_step = 0
    this_min = 0
    this_max = 2
    print('reference cluster number',n_cluster)
    while this_step < max_steps:
        this_resolution = this_min + ((this_max-this_min)/2)
        sc.tl.leiden(embadata,resolution=this_resolution,random_state=42)
        this_clusters = embadata.obs['leiden'].nunique()
        if this_clusters > n_cluster:
            this_max = this_resolution
        elif this_clusters < n_cluster:
            this_min = this_resolution
        else:break
        this_step+=1
    if this_step==max_steps:
        print('Cannot find the number of clusters')
        print('Use resolution',this_resolution)
    else:
        print('use resolution',this_resolution)
     # leiden
    sc.tl.leiden(embadata,resolution=this_resolution,random_state=42,key_added=f'cluster_{n_cluster}')

## Baron

In [ ]:
refdf = pd.read_csv('./data/baron_human_ref_19264_fromsaver.csv',index_col=0)

In [ ]:
refAdata = sc.AnnData(refdf)
refAdata = postprocess(refAdata,True)

In [ ]:
sc.tl.leiden(refAdata,resolution=0.6)

In [ ]:
numcls = refAdata.obs.leiden.unique().shape[0]
numcls

In [ ]:
rcParams['axes.spines.right'] = False
rcParams['axes.spines.top'] = False
rcParams['pdf.fonttype'] = 42
rcParams['ps.fonttype'] = 42

In [ ]:
sc.pl.umap(refAdata,color=[f'leiden'],size=25,title='Reference',legend_loc='on data',save='reference')

### concat embedding

In [ ]:
imputeemb = np.load(f'../../imputation/SAVER/baron/baron_human_samp_19264_fromsaver_50M-0.1B-res_tgthighres5_embedding.npy')
imputeAdata = sc.AnnData(pd.DataFrame(imputeemb,index=refdf.index))
sc.pp.scale(imputeAdata)
sc.tl.pca(imputeAdata)
sc.pp.neighbors(imputeAdata)
sc.tl.umap(imputeAdata)
cluster_k_leiden(imputeAdata,numcls)
imputeAdata.obs['refleiden']=refAdata.obs['leiden']

In [ ]:
figsize(5,5)
sc.pl.umap(imputeAdata,color=[f'refleiden'],size=25,title='Reference',legend_loc='on data',save='concat')

### max embedding

In [ ]:
maxembAdata = sc.AnnData(pd.DataFrame(imputeemb[:,2*768:3*768],index=refdf.index))
sc.pp.scale(maxembAdata)
sc.tl.pca(maxembAdata)
sc.pp.neighbors(maxembAdata)
sc.tl.umap(maxembAdata)
cluster_k_leiden(maxembAdata,numcls)
maxembAdata.obs['refleiden']=refAdata.obs['leiden']

In [ ]:
sc.pl.umap(maxembAdata,color=[f'refleiden'],size=25,title='Reference',legend_loc='on data',save='max')

### mean embedding

In [ ]:
meanembAdata = sc.AnnData(pd.DataFrame(imputeemb[:,3*768:4*768],index=refdf.index))
sc.pp.scale(meanembAdata)
sc.tl.pca(meanembAdata)
sc.pp.neighbors(meanembAdata)
sc.tl.umap(meanembAdata)
cluster_k_leiden(meanembAdata,numcls)
meanembAdata.obs['refleiden']=refAdata.obs['leiden']

In [ ]:
sc.pl.umap(meanembAdata,color=[f'refleiden'],size=25,title='Reference',legend_loc='on data',save='mean')

### S embedding

In [ ]:
sembAdata = sc.AnnData(pd.DataFrame(imputeemb[:,0*768:1*768],index=refdf.index))
sc.pp.scale(sembAdata)
sc.tl.pca(sembAdata)
sc.pp.neighbors(sembAdata)
sc.tl.umap(sembAdata)
cluster_k_leiden(sembAdata,numcls)
sembAdata.obs['refleiden']=refAdata.obs['leiden']

In [ ]:
sc.pl.umap(sembAdata,color=[f'refleiden'],size=25,title='Reference',legend_loc='on data',save='token_S')

In [ ]:
from sklearn.metrics.cluster import adjusted_rand_score,normalized_mutual_info_score,davies_bouldin_score,calinski_harabasz_score,silhouette_score

alladata = [imputeAdata,maxembAdata,meanembAdata,sembAdata]
name = ['concat','max','mean','token_S']
allscore=[]
for testadata in alladata:
    imputenmi = normalized_mutual_info_score(refAdata.obs[f'leiden'], testadata.obs[f'cluster_7']) 
    imputeari = adjusted_rand_score(refAdata.obs[f'leiden'], testadata.obs[f'cluster_7']) 
    imputesil = silhouette_score(refAdata.obsm['X_pca'],testadata.obs[f'cluster_7'])
    allscore.append([imputenmi,imputeari,imputesil])

In [ ]:
results = pd.DataFrame(allscore,index=name,columns=['nmi','ari','sil'])
np.round(results,3)

In [ ]:
for i in range(4):
    alladata[i].write_h5ad(f'baron/{name[i]}.h5ad')

In [ ]:
refAdata.write_h5ad('baron/reference.h5ad')

In [ ]:
results.to_csv('baron_results.csv')

## Zheng68K

In [ ]:
label = np.load('./data/zheng68kall_label.npy',allow_pickle=True)

In [ ]:
refAdata = sc.read_h5ad('./data/zheng68k_reference.h5ad')

### concat embedding

In [ ]:
numcls=10
imputeemb = np.load(f'./data/pbmc68ksorted_count_50M-0.1B-res_tgthighres1.0_embedding.npy')
imputeAdata = sc.AnnData(imputeemb)
sc.pp.scale(imputeAdata)
sc.tl.pca(imputeAdata)
sc.pp.neighbors(imputeAdata)
cluster_k_leiden(imputeAdata,numcls)
imputeAdata.obs['refleiden']=label

In [ ]:
numcls=10
imputeemb = np.load(f'./data/pbmc68ksorted_count_50M-0.1B-res_tgthighres1.0_embedding.npy')
imputeAdata = sc.AnnData(imputeemb)
sc.pp.scale(imputeAdata)
sc.tl.pca(imputeAdata)
sc.pp.neighbors(imputeAdata)
cluster_k_leiden(imputeAdata,numcls)
imputeAdata.obs['refleiden']=label

### max embedding

In [ ]:
maxembAdata = sc.AnnData(imputeemb[:,2*768:3*768])
sc.pp.scale(maxembAdata)
sc.tl.pca(maxembAdata)
sc.pp.neighbors(maxembAdata)
cluster_k_leiden(maxembAdata,numcls)
maxembAdata.obs['refleiden']=label

### mean embedding

In [ ]:
meanembAdata = sc.AnnData(imputeemb[:,3*768:4*768])
sc.pp.scale(meanembAdata)
sc.tl.pca(meanembAdata)
sc.pp.neighbors(meanembAdata)
cluster_k_leiden(meanembAdata,numcls)
meanembAdata.obs['refleiden']=label

### S embedding

In [ ]:
sembAdata = sc.AnnData(imputeemb[:,0*768:1*768])
sc.pp.scale(sembAdata)
sc.tl.pca(sembAdata)
sc.pp.neighbors(sembAdata)
cluster_k_leiden(sembAdata,numcls)
sembAdata.obs['refleiden']=label

In [ ]:
from sklearn.metrics.cluster import adjusted_rand_score,normalized_mutual_info_score,davies_bouldin_score,calinski_harabasz_score,silhouette_score

alladata = [imputeAdata,maxembAdata,meanembAdata,sembAdata]
name = ['concat','max','mean','token_S']
allscore=[]
for testadata in alladata:
    imputenmi = normalized_mutual_info_score(label, testadata.obs[f'cluster_10']) 
    imputeari = adjusted_rand_score(label, testadata.obs[f'cluster_10']) 
    imputesil = silhouette_score(refAdata.obsm['X_pca'],testadata.obs[f'cluster_10'])
    allscore.append([imputenmi,imputeari,imputesil])

In [ ]:
results = pd.DataFrame(allscore,index=name,columns=['nmi','ari','sil'])
np.round(results,3)

In [ ]:
for i in range(4):
    alladata[i].write_h5ad(f'zheng68k/{name[i]}.h5ad')

In [ ]:
refAdata.write_h5ad('zheng68k/reference.h5ad')

In [ ]:
results.to_csv('zheng_results.csv')